In [3]:
import pandas as pd
import xarray as xr

# 1. Laad beide bestanden in
ds_instant = xr.open_dataset('C:\\Users\\qsoerohardjo\\Documents\\GitHub\\datalab-v\\data\\processed\\data_stream-oper_stepType-instant.nc')
ds_accum = xr.open_dataset('C:\\Users\\qsoerohardjo\\Documents\\GitHub\\datalab-v\\data\\processed\\data_stream-oper_stepType-accum.nc')

# 2. Voeg ze samen tot één dataset
# xarray koppelt ze automatisch op basis van tijd, latitude en longitude
ds_combined = xr.merge([ds_instant, ds_accum])

# 3. Bekijk het resultaat
print(ds_combined)

# Nu kun je door naar de DataFrame stap:
df = ds_combined.to_dataframe().reset_index()

C:\Users\qsoerohardjo\AppData\Local\Temp\ipykernel_7828\4119827273.py:10: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_combined = xr.merge([ds_instant, ds_accum])
C:\Users\qsoerohardjo\AppData\Local\Temp\ipykernel_7828\4119827273.py:10: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_combined = xr.merge([ds_instant, ds_accum])


<xarray.Dataset> Size: 460MB
Dimensions:     (valid_time: 2160, latitude: 81, longitude: 73)
Coordinates:
    number      int64 8B 0
  * valid_time  (valid_time) datetime64[ns] 17kB 2000-06-01 ... 2002-06-30T23...
  * latitude    (latitude) float64 648B 15.0 14.75 14.5 ... -4.5 -4.75 -5.0
  * longitude   (longitude) float64 584B 33.0 33.25 33.5 ... 50.5 50.75 51.0
    expver      (valid_time) <U4 35kB '0001' '0001' '0001' ... '0001' '0001'
Data variables:
    u10         (valid_time, latitude, longitude) float32 51MB ...
    v10         (valid_time, latitude, longitude) float32 51MB ...
    d2m         (valid_time, latitude, longitude) float32 51MB ...
    t2m         (valid_time, latitude, longitude) float32 51MB ...
    msl         (valid_time, latitude, longitude) float32 51MB ...
    swvl1       (valid_time, latitude, longitude) float32 51MB ...
    tp          (valid_time, latitude, longitude) float32 51MB ...
    e           (valid_time, latitude, longitude) float32 51MB ...
    

In [7]:
df_cum = ds_accum.to_dataframe().reset_index()

In [8]:
df_cum

,valid_time,latitude,longitude,tp,e,ro,number,expver
0,2000-06-01 00:00:00,15.0,33.00,0.000000,-3.958354e-06,0.0,0,0001
1,2000-06-01 00:00:00,15.0,33.25,0.000000,-1.033605e-05,0.0,0,0001
2,2000-06-01 00:00:00,15.0,33.50,0.000000,-7.695053e-07,0.0,0,0001
3,2000-06-01 00:00:00,15.0,33.75,0.000000,-4.703412e-06,0.0,0,0001
4,2000-06-01 00:00:00,15.0,34.00,0.000000,-4.416797e-07,0.0,0,0001
...,...,...,...,...,...,...,...,...
12772075,2002-06-30 23:00:00,-5.0,50.00,0.000064,-1.811383e-04,0.0,0,0001
12772076,2002-06-30 23:00:00,-5.0,50.25,0.000068,-1.776216e-04,0.0,0,0001
12772077,2002-06-30 23:00:00,-5.0,50.50,0.000099,-1.789925e-04,0.0,0,0001
12772078,2002-06-30 23:00:00,-5.0,50.75,0.000093,-1.880822e-04,0.0,0,0001


In [9]:
df_instant = ds_instant.to_dataframe().reset_index()
display(df_instant.head())

,valid_time,latitude,longitude,u10,v10,d2m,t2m,msl,swvl1,number,expver
0,2000-06-01,15.0,33.00,0.506241,4.229965,289.776855,304.123779,100593.3125,0.185074,0,0001
1,2000-06-01,15.0,33.25,0.513077,3.790512,290.270996,303.905029,100600.0625,0.193344,0,0001
2,2000-06-01,15.0,33.50,0.538467,3.432114,290.360840,303.602295,100611.5625,0.200699,0,0001
3,2000-06-01,15.0,33.75,0.746475,2.830551,290.604980,303.049561,100613.3125,0.205414,0,0001
4,2000-06-01,15.0,34.00,1.061905,2.838364,290.622559,302.928467,100619.8125,0.183578,0,0001


In [2]:
# Kelvin naar Celsius
df['t2m'] = df['t2m'] - 273.15
df['d2m'] = df['d2m'] - 273.15

# Meters naar millimeters (tp, e, ro)
# Let op: e (verdamping) is vaak negatief in ERA5, we maken het positief voor gemak
df['tp'] = df['tp'] * 1000
df['e'] = df['e'].abs() * 1000
df['ro'] = df['ro'] * 1000

In [3]:
df = df.sort_values(by=['latitude', 'longitude', 'valid_time']).reset_index(drop=True)

In [4]:
def fast_api(precip_array, decay=0.85):
    api = [0.0] * len(precip_array)
    for i in range(1, len(precip_array)):
        api[i] = (api[i-1] * decay) + precip_array[i]
    return api

# We groeperen per locatie en passen de API toe
df['api'] = df.groupby(['latitude', 'longitude'])['tp'].transform(lambda x: fast_api(x.values))

In [5]:
# Bepaal de drempelwaarde (bijv. 99e percentiel)
threshold = df['tp'].quantile(0.99)

# Maak de target: is de neerslag over 24 uur groter dan de drempel?
df['target_flood'] = df.groupby(['latitude', 'longitude'])['tp'].shift(-24)
df['target_flood'] = (df['target_flood'] > threshold).astype(int)

# Verwijder de laatste 24 uur per locatie (die hebben geen target)
df = df.dropna(subset=['target_flood'])

In [6]:
# Groepeer per locatie om te voorkomen dat data van verschillende plekken mengt
grouped = df.groupby(['latitude', 'longitude'])

# 1. Regen van de afgelopen 6 uur (Trigger intensiteit)
df['tp_6h_sum'] = grouped['tp'].transform(lambda x: x.rolling(window=6).sum())

# 2. Regen van de afgelopen week (Bodemverzadiging)
# Omdat we uurlijkse data hebben: 7 dagen * 24 uur = 168
df['tp_7d_sum'] = grouped['tp'].transform(lambda x: x.rolling(window=168).sum())

# 3. Temperatuur trend (helpt bij verdamping/uitdroging)
df['t2m_24h_avg'] = grouped['t2m'].transform(lambda x: x.rolling(window=24).mean())

# Voorspel de status van over 2 weken
df['target_flood_2w'] = df.groupby(['latitude', 'longitude'])['tp'].shift(-336)

# Vergeet niet de NaN waardes te droppen die ontstaan aan het begin van de reeks
df = df.dropna()

In [7]:
# 1. Splits de data in overstromingen en niet-overstromingen
df_floods = df[df['target_flood'] == 1]
df_no_floods = df[df['target_flood'] == 0]

# 2. Pak een subset van de normale dagen (bijv. 2x zoveel als de overstromingen)
df_no_floods_sample = df_no_floods.sample(n=len(df_floods) * 2, random_state=42)

# 3. Voeg ze weer samen tot een trainingsset
df_train_balanced = pd.concat([df_floods, df_no_floods_sample])

# 4. Schud de data door elkaar
df_train_balanced = df_train_balanced.sample(frac=1, random_state=42)

print(f"Nieuwe dataset grootte: {len(df_train_balanced)} rijen")
print(df_train_balanced['target_flood'].value_counts(normalize=True))

Nieuwe dataset grootte: 303567 rijen
target_flood
0    0.666667
1    0.333333
Name: proportion, dtype: float64


In [18]:
# Coördinaten box voor Ethiopië
ethiopia_lat = (3, 15)
ethiopia_lon = (33, 48)

df_eth = df[
    (df['latitude'] >= ethiopia_lat[0]) & (df['latitude'] <= ethiopia_lat[1]) &
    (df['longitude'] >= ethiopia_lon[0]) & (df['longitude'] <= ethiopia_lon[1])
].copy()

In [38]:
# Maak een datum-kolom (zonder uren)
df_eth['date'] = df_eth['valid_time'].dt.date

# Aggregeren: Som voor neerslag/runoff, gemiddelde voor de rest
df_daily = df_eth.groupby(['latitude', 'longitude', 'date']).agg({
    'tp': 'sum',
    'ro': 'sum',
    'e': 'sum',
    't2m': 'mean',
    'msl': 'mean',
    'swvl1': 'mean',
    'u10': 'mean',
    'v10': 'mean'
}).reset_index()

# Sorteer op tijd voor de rolling windows
df_daily = df_daily.sort_values(['latitude', 'longitude', 'date'])

In [39]:
# Bepaal drempelwaarde voor dagelijkse neerslag (bijv. 95e percentiel)
daily_threshold = df_daily['tp'].quantile(0.95)

# Shift de target met 14 dagen
df_daily['target_flood_14d'] = df_daily.groupby(['latitude', 'longitude'])['tp'].shift(-14)
df_daily['target_flood_14d'] = (df_daily['target_flood_14d'] > daily_threshold).astype(int)

In [40]:
len(df_daily[df_daily['target_flood_14d'] == 1])

9192

In [45]:
grouped = df_daily.groupby(['latitude', 'longitude'])

# Kortere termijn voorgeschiedenis
df_daily['tp_7d_sum'] = grouped['tp'].transform(lambda x: x.rolling(window=7, min_periods=1).sum())
df_daily['tp_14d_sum'] = grouped['tp'].transform(lambda x: x.rolling(window=14, min_periods=1).sum())

# Update je feature lijst voor de training
features_daily = ['t2m', 'msl', 'swvl1', 'tp_7d_sum', 'tp_14d_sum', 'u10', 'v10']

In [47]:
df_daily

,latitude,longitude,date,tp,ro,e,t2m,msl,swvl1,u10,v10,target_flood_14d,tp_30d_sum,tp_90d_sum,tp_7d_sum,tp_14d_sum
0,3.0,33.0,2000-06-07,0.460625,0.004768,0.012562,19.958893,101418.687500,0.358828,-1.174774,-0.192230,0,NaN,NaN,0.460625,0.460625
1,3.0,33.0,2000-06-08,7.514477,0.081301,3.292880,22.788544,101423.101562,0.353009,-0.492668,-0.312014,0,NaN,NaN,7.975101,7.975101
2,3.0,33.0,2000-06-09,8.419037,0.097752,3.562241,23.392242,101521.242188,0.352143,0.002078,-0.551365,0,NaN,NaN,16.394138,16.394138
3,3.0,33.0,2000-06-10,1.560211,0.035286,3.562676,23.317434,101477.460938,0.364978,-0.434367,-0.112125,1,NaN,NaN,17.954350,17.954350
4,3.0,33.0,2000-06-11,0.092506,0.023365,3.612488,24.926880,101324.117188,0.328925,-1.144711,0.194663,0,NaN,NaN,18.046856,18.046856
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209225,15.0,48.0,2002-06-12,0.000000,0.000000,0.046960,27.386810,100502.187500,0.006449,-0.083423,-1.034515,0,6.151199,NaN,0.226021,1.905441
209226,15.0,48.0,2002-06-13,0.000000,0.000000,0.051271,27.465616,100363.351562,0.005187,-0.261326,-1.197215,0,6.151199,NaN,0.004292,1.905441
209227,15.0,48.0,2002-06-14,1.808643,0.006676,0.221096,24.079905,100528.835938,0.017904,1.356000,4.492678,0,7.959843,NaN,1.808643,3.690243
209228,15.0,48.0,2002-06-15,0.000477,0.000000,0.303162,27.392487,100351.750000,0.024777,0.829041,0.056276,0,7.960320,NaN,1.809120,3.690720


In [48]:
# Hoeveel bruikbare targets hebben we per jaar?
df_daily['date'] = pd.to_datetime(df_daily['date'])

print(df_daily.dropna(subset=['target_flood_14d']).groupby(df_daily['date'].dt.year).size())

date
2000    71736
2001    89670
2002    47824
dtype: int64


In [51]:
# Verwijder de rijen die door de shift van 14 dagen op NaN komen te staan (de tweede helft van elke juni)
df_model_ready = df_daily.dropna(subset=['target_flood_14d']).copy()

# Splitsen op tijd (2000-2001 train, 2002 test)
train_mask = df_model_ready['date'].dt.year < 2002
test_mask = df_model_ready['date'].dt.year == 2002

X_train = df_model_ready.loc[train_mask, features_daily]
y_train = df_model_ready.loc[train_mask, 'target_flood_14d']

X_test = df_model_ready.loc[test_mask, features_daily]
y_test = df_model_ready.loc[test_mask, 'target_flood_14d']

# Balanceren (zoals we eerder deden)
train_tmp = pd.concat([X_train, y_train], axis=1)
floods = train_tmp[train_tmp['target_flood_14d'] == 1]
no_floods = train_tmp[train_tmp['target_flood_14d'] == 0].sample(n=len(floods)*3, random_state=42)
df_balanced = pd.concat([floods, no_floods]).sample(frac=1)

# Model trainen
model_eth = XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=4)
model_eth.fit(df_balanced[features_daily], df_balanced['target_flood_14d'])

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [52]:
# Voorspellen op de testset (2002)
y_probs = model_eth.predict_proba(X_test)[:, 1]
y_pred = (y_probs > 0.5).astype(int)

print("\n--- Resultaten Ethiopië 14-daagse Voorspelling ---")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_probs):.4f}")


--- Resultaten Ethiopië 14-daagse Voorspelling ---
              precision    recall  f1-score   support

           0       1.00      0.92      0.96     47473
           1       0.05      0.49      0.08       351

    accuracy                           0.92     47824
   macro avg       0.52      0.71      0.52     47824
weighted avg       0.99      0.92      0.95     47824

ROC-AUC Score: 0.8905
